# NOTES

Since "Reference - Global Baseline Model - Linear Regression & Support Vector Machine" **failed**, based on the clustering + PCA as a feature selection steps, the clusters ("LEADERS" & "FOLLOWERS") were was trained on the following models -

**1. Baseline Model:** a. Linear Regression, b. Support Vector Regressor
    
**2. Timeseries Model:** a. ARIMAX (ARIMA + exogenuous variables)

In [1]:
import os
import joblib
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.svm import SVR
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from statsmodels.tsa.stattools import arma_order_select_ic
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [2]:
# Import dataset
dataset = pd.read_csv(r"C:\Users\Disha\TU Braunschweig\Thesis\Data_2025\Cleaned Dataset.csv")
dataset

,Country,Year,BEV Percentage (Total Number Of Registrations),GDP,CPI,EG,Recharging Points,AC Recharging Speed (km/h),DC Recharging Speed (km/h),Available,Battery Capacity,Real Range,Purchase price (EUR),Log_BEV Percentage (Total Number Of Registrations),Log_Recharging Points,Log_GDP,Log_CPI,Log_EG,Log_Available,Log_AC Recharging Speed (km/h)
0,Austria,2020,6.41,3.803179e+05,1.381911,-1.600,26797.00000,39.325301,403.164557,97.000000,57.761446,286.200000,38311.380000,2.002830,10.196083,12.848765,0.867903,NaN,4.584967,3.696979
1,Austria,2021,13.92,4.062315e+05,2.766667,2.000,39904.00000,40.904348,493.391304,168.000000,64.046957,308.730000,40745.010000,2.702703,10.594257,12.914681,1.326190,1.098612,5.129899,3.735390
2,Austria,2022,15.70,4.493822e+05,8.546870,2.700,61393.00000,49.314815,594.259259,253.000000,73.787037,345.440000,43874.270000,2.815409,11.025067,13.015631,2.256213,1.308333,5.537334,3.918300
3,Austria,2023,19.91,4.778373e+05,7.814134,0.800,76773.00000,70.307692,649.230769,371.000000,79.230769,379.510785,45999.050000,3.040228,11.248621,13.077028,2.176357,0.587787,5.918894,4.267004
4,Austria,2024,17.58,4.940876e+05,2.937916,0.100,102562.00000,92.176355,744.736940,551.323143,88.123464,416.941976,48874.922228,2.922086,11.538233,13.110470,1.370652,0.095310,6.314133,4.534494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,Sweden,2021,19.09,5.417760e+06,2.163197,1.300,70537.00000,40.904348,493.391304,168.000000,64.046957,308.730000,40745.010000,3.000222,11.163907,15.505193,1.151583,0.832909,5.129899,3.735390
158,Sweden,2022,32.84,5.816415e+06,8.369291,3.500,83510.00000,49.314815,594.259259,253.000000,73.787037,345.440000,43874.270000,3.521644,11.332734,15.576195,2.237437,1.504077,5.537334,3.918300
159,Sweden,2023,38.42,6.143187e+06,8.548625,1.200,126758.00000,70.307692,649.230769,371.000000,79.230769,379.510785,45999.050000,3.674273,11.750043,15.630854,2.256397,0.788457,5.918894,4.267004
160,Sweden,2024,34.99,6.379843e+06,2.835817,-0.300,189977.00000,92.176355,744.736940,551.323143,88.123464,416.941976,48874.922228,3.583241,12.154664,15.668654,1.344382,-0.356675,6.314133,4.534494


In [3]:
# Define the target
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [4]:
# Load the PCA tools
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

In [5]:
# Split the dataset into train and test data
pre_covid_df = dataset[dataset['Year'].between(2020, 2024)].copy()
post_covid_df = dataset[dataset['Year'] >= 2025].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [6]:
# Check which columns have NaNs and how many
print("Missing Values Per Column on Train data (2020-2024)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column Test data (2025)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column on Train data (2020-2024)
DC Recharging Speed (km/h)         0
Log_Recharging Points              1
Real Range                         0
Purchase price (EUR)               0
Log_Available                      0
Log_AC Recharging Speed (km/h)     0
Battery Capacity                   0
Log_GDP                            0
Log_CPI                            1
Log_EG                            22
dtype: int64

Missing Values Per Column Test data (2025)
DC Recharging Speed (km/h)        0
Log_Recharging Points             0
Real Range                        0
Purchase price (EUR)              0
Log_Available                     0
Log_AC Recharging Speed (km/h)    0
Battery Capacity                  0
Log_GDP                           0
Log_CPI                           0
Log_EG                            0
dtype: int64


In [7]:
# Checking which rows are empty
print("Missing Log_EG in Train data (2020-2024)")
print(pre_covid_df[pre_covid_df['Log_EG'].isna()].reset_index()[['Country', 'Year', 'Log_EG']])

Missing Log_EG in Train data (2020-2024)
           Country  Year  Log_EG
0          Austria  2020     NaN
1         Bulgaria  2020     NaN
2          Croatia  2020     NaN
3   Czech Republic  2020     NaN
4          Denmark  2020     NaN
5          Estonia  2020     NaN
6          Finland  2020     NaN
7          Finland  2024     NaN
8           Greece  2020     NaN
9          Hungary  2020     NaN
10         Hungary  2024     NaN
11         Ireland  2020     NaN
12           Italy  2020     NaN
13          Latvia  2021     NaN
14          Latvia  2024     NaN
15       Lithuania  2020     NaN
16        Portugal  2020     NaN
17         Romania  2020     NaN
18         Romania  2023     NaN
19        Slovakia  2020     NaN
20           Spain  2020     NaN
21          Sweden  2020     NaN


In [8]:
features_to_impute = ['Log_Recharging Points', 'Log_EG', 'Log_CPI']

# Compute per-country median from train
train_country_medians = {}
train_global_medians = {}

for col in features_to_impute:
    train_country_medians[col] = pre_covid_df.groupby('Country')[col].median()
    train_global_medians[col] = pre_covid_df[col].median()

def robust_impute(df, country_medians, global_medians):
    df_copy = df.copy()
    for col in features_to_impute:
        
        # Impute per-country median
        df_copy[col] = df_copy[col].fillna(df_copy['Country'].map(country_medians[col]))
        
        # Fallback to global median if still NaN
        df_copy[col] = df_copy[col].fillna(global_medians[col])
    return df_copy

# Apply to train and test
pre_covid_fixed = robust_impute(pre_covid_df, train_country_medians, train_global_medians)
post_covid_fixed = robust_impute(post_covid_df, train_country_medians, train_global_medians)

In [9]:
# Verfication
print("Missing Values Per Column on Train data (2020-2024)")
print(pre_covid_fixed[features].isna().sum())

print("\nMissing Values Per Column Test data (2025)")
print(post_covid_fixed[features].isna().sum())

Missing Values Per Column on Train data (2020-2024)
DC Recharging Speed (km/h)        0
Log_Recharging Points             0
Real Range                        0
Purchase price (EUR)              0
Log_Available                     0
Log_AC Recharging Speed (km/h)    0
Battery Capacity                  0
Log_GDP                           0
Log_CPI                           0
Log_EG                            0
dtype: int64

Missing Values Per Column Test data (2025)
DC Recharging Speed (km/h)        0
Log_Recharging Points             0
Real Range                        0
Purchase price (EUR)              0
Log_Available                     0
Log_AC Recharging Speed (km/h)    0
Battery Capacity                  0
Log_GDP                           0
Log_CPI                           0
Log_EG                            0
dtype: int64


In [10]:
X_train_scaled = scaler.transform(pre_covid_fixed[features])
X_test_scaled = scaler.transform(post_covid_fixed[features])

# Convert back to DataFrame to keep column names 
X_train_scaled = pd.DataFrame(X_train_scaled, columns=features)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=features)

In [11]:
# Transform the features into 3 Principal Components using the loaded PCA model
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2', 'PC3'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2', 'PC3'], index=post_covid_fixed.index)

In [12]:
# Define the targets
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (135, 3)
Test Shape (PCs): (27, 3)


In [13]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    "Austria", "Belgium", "Czech Republic", "Denmark", "France", 
    "Germany", "Hungary", "Italy", "Netherlands", "Poland", 
    "Romania", "Spain", "Sweden"
]

followers_list = [
    "Bulgaria", "Croatia", "Cyprus", "Estonia", "Finland", 
    "Greece", "Ireland", "Latvia", "Lithuania", "Luxembourg", 
    "Malta", "Portugal", "Slovakia", "Slovenia"
]

In [14]:
# Filter Training Data 
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data 
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## Baseline Models

**1. Linear Regression**

In [15]:
def run_lr_model(X_train, y_train, X_test, y_test, cluster_name):
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    train_pred = lr.predict(X_train)
    test_pred = lr.predict(X_test)

    # Back-transform
    y_train_real = np.exp(y_train)
    y_test_real = np.exp(y_test)
    train_pred_real = np.exp(train_pred)
    test_pred_real = np.exp(test_pred)

    # Compute metrics
    r2_train_log = r2_score(y_train, train_pred)
    r2_test_log = r2_score(y_test, test_pred)

    r2_train_real = r2_score(y_train_real, train_pred_real)
    r2_test_real = r2_score(y_test_real, test_pred_real)

    mae_train_real = mean_absolute_error(y_train_real, train_pred_real)
    mae_test_real = mean_absolute_error(y_test_real, test_pred_real)

    print(f"{cluster_name} - Linear Regression")
    print(f"Train R² (Log): {r2_train_log:.4f}, R² (Real): {r2_train_real:.4f}, MAE (Real): {mae_train_real:.2f}%")
    print(f"Test  R² (Log): {r2_test_log:.4f}, R² (Real): {r2_test_real:.4f}, MAE (Real): {mae_test_real:.2f}%\n")

    return {
        'lr_model': lr,
        'y_test_real': y_test_real,
        'pred_test_real': test_pred_real
    }

In [16]:
# Run for both clusters
results_lr_leaders = run_lr_model(X_train_leaders, y_train_leaders, X_test_leaders, y_test_leaders, "Leaders")
results_lr_followers = run_lr_model(X_train_followers, y_train_followers, X_test_followers, y_test_followers, "Followers")

Leaders - Linear Regression
Train R² (Log): 0.2426, R² (Real): 0.1255, MAE (Real): 7.59%
Test  R² (Log): 0.1002, R² (Real): 0.0743, MAE (Real): 13.26%

Followers - Linear Regression
Train R² (Log): 0.4370, R² (Real): 0.2345, MAE (Real): 4.23%
Test  R² (Log): -0.3312, R² (Real): -0.1682, MAE (Real): 9.61%



In [17]:
# Combine predictions and actual values for leaders and followers
combined_preds = np.concatenate([
    results_lr_leaders['pred_test_real'],
    results_lr_followers['pred_test_real']
])

combined_actuals = pd.concat([
    results_lr_leaders['y_test_real'],
    results_lr_followers['y_test_real']
])

# Combine metadata in the same order
combined_metadata = pd.concat([
    post_covid_fixed[post_covid_fixed['Country'].isin(leaders_list)],
    post_covid_fixed[post_covid_fixed['Country'].isin(followers_list)]
])

# Build final results table
all_lr_results = pd.DataFrame({
    'Country': combined_metadata['Country'].values,
    'Year': combined_metadata['Year'].values,
    'Actual_BEV_%': combined_actuals.values,
    'Predicted_BEV_%': combined_preds
})

# Absolute error
all_lr_results['Error_Abs'] = (
    all_lr_results['Actual_BEV_%'] - all_lr_results['Predicted_BEV_%']
).abs()

# Print results by year and cluster
for year in sorted(all_lr_results['Year'].unique()):
    print(f"\n---> LINEAR REGRESSION: {int(year)} PREDICTION RESULTS")

    year_data = all_lr_results[all_lr_results['Year'] == year]

    leaders_data = (
        year_data[year_data['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )

    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))

    followers_data = (
        year_data[year_data['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )

    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))


---> LINEAR REGRESSION: 2025 PREDICTION RESULTS

**** LEADERS CLUSTER (2025) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
       Denmark         69.53        21.264018  48.265982
   Netherlands         37.68        22.109169  15.570831
        Sweden         37.58        25.333253  12.246747
       Belgium         35.40        20.055107  15.344893
       Austria         22.34        17.042929   5.297071
        France         21.01        24.425842   3.415842
       Germany         20.08        27.989822   7.909822
         Spain         10.19        17.727099   7.537099
       Hungary          9.36        25.808005  16.448005
        Poland          7.73        15.058363   7.328363
         Italy          7.18        19.745989  12.565989
       Romania          6.59        14.158107   7.568107
Czech Republic          6.55        19.427416  12.877416

**** FOLLOWERS CLUSTER (2025) ****
   Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Finland         37.81      

In [18]:
# Save the results in an excel file

# Filter the results as per the clusters
all_lr_results['Cluster'] = all_lr_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_df = (
    all_lr_results[all_lr_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_df = (
    all_lr_results[all_lr_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_df.to_excel("Trial3_LR_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial3_LR_Followers_Results.xlsx", index=False)

**2. Support Vector Machine**

In [19]:
def run_svr_model(X_train, y_train, X_test, y_test, cluster_name):
    svr = SVR(kernel='rbf', C=100, epsilon=0.1)
    svr.fit(X_train, y_train)

    train_pred = svr.predict(X_train)
    test_pred = svr.predict(X_test)

    # Back-transform
    y_train_real = np.exp(y_train)
    y_test_real = np.exp(y_test)
    train_pred_real = np.exp(train_pred)
    test_pred_real = np.exp(test_pred)

    # Compute metrics
    r2_train_log = r2_score(y_train, train_pred)
    r2_test_log = r2_score(y_test, test_pred)

    r2_train_real = r2_score(y_train_real, train_pred_real)
    r2_test_real = r2_score(y_test_real, test_pred_real)

    mae_train_real = mean_absolute_error(y_train_real, train_pred_real)
    mae_test_real = mean_absolute_error(y_test_real, test_pred_real)

    print(f"{cluster_name} - Support Vector Regression")
    print(f"Train R² (Log): {r2_train_log:.4f}, R² (Real): {r2_train_real:.4f}, MAE (Real): {mae_train_real:.2f}%")
    print(f"Test  R² (Log): {r2_test_log:.4f}, R² (Real): {r2_test_real:.4f}, MAE (Real): {mae_test_real:.2f}%\n")

    return {
        'svr_model': svr,
        'y_test_real': y_test_real,
        'pred_test_real': test_pred_real
    }

In [20]:
# Run for both clusters
results_svr_leaders = run_svr_model(X_train_leaders, y_train_leaders, X_test_leaders, y_test_leaders, "Leaders")
results_svr_followers = run_svr_model(X_train_followers, y_train_followers, X_test_followers, y_test_followers, "Followers")

Leaders - Support Vector Regression
Train R² (Log): 0.4639, R² (Real): 0.2864, MAE (Real): 5.69%
Test  R² (Log): -3.2580, R² (Real): -8.3131, MAE (Real): 49.58%

Followers - Support Vector Regression
Train R² (Log): 0.7354, R² (Real): 0.7157, MAE (Real): 2.23%
Test  R² (Log): -8.6410, R² (Real): -1.4380, MAE (Real): 13.66%



In [21]:
# Combine predictions and actual values for leaders and followers
combined_preds = np.concatenate([
    results_svr_leaders['pred_test_real'],
    results_svr_followers['pred_test_real']
])

combined_actuals = pd.concat([
    results_svr_leaders['y_test_real'],
    results_svr_followers['y_test_real']
])

# Combine metadata in the same order
combined_metadata = pd.concat([
    post_covid_fixed[post_covid_fixed['Country'].isin(leaders_list)],
    post_covid_fixed[post_covid_fixed['Country'].isin(followers_list)]
])

# Build final results table
all_svr_results = pd.DataFrame({
    'Country': combined_metadata['Country'].values,
    'Year': combined_metadata['Year'].values,
    'Actual_BEV_%': combined_actuals.values,
    'Predicted_BEV_%': combined_preds
})

# Absolute error
all_svr_results['Error_Abs'] = (
    all_svr_results['Actual_BEV_%'] - all_svr_results['Predicted_BEV_%']
).abs()

# Print results by year and cluster
for year in sorted(all_svr_results['Year'].unique()):
    print(f"\n---> SUPPORT VECTOR REGRESSION: {int(year)} PREDICTION RESULTS")

    year_data = all_svr_results[all_svr_results['Year'] == year]

    leaders_data = (
        year_data[year_data['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )

    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))

    followers_data = (
        year_data[year_data['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )

    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))


---> SUPPORT VECTOR REGRESSION: 2025 PREDICTION RESULTS

**** LEADERS CLUSTER (2025) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
       Denmark         69.53        90.383411  20.853411
   Netherlands         37.68        95.339420  57.659420
        Sweden         37.58        49.880938  12.300938
       Belgium         35.40        78.596455  43.196455
       Austria         22.34        73.017185  50.677185
        France         21.01        80.300494  59.290494
       Germany         20.08        70.628990  50.548990
         Spain         10.19        67.180570  56.990570
       Hungary          9.36        86.075293  76.715293
        Poland          7.73        20.482490  12.752490
         Italy          7.18        87.994643  80.814643
       Romania          6.59        49.476839  42.886839
Czech Republic          6.55        86.355474  79.805474

**** FOLLOWERS CLUSTER (2025) ****
   Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Finland         37.

In [22]:
# Save the results in an excel file

# Filter the results as per the clusters
all_svr_results['Cluster'] = all_svr_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_df = (
    all_svr_results[all_svr_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_df = (
    all_svr_results[all_svr_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_df.to_excel("Trial3_SVR_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial3_SVR_Followers_Results.xlsx", index=False)

## Time-Series Models

**3. ARIMAX (with exogenous variables)**

In [23]:
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter('ignore', UserWarning)

def run_arimax_model(X_train, y_train, X_test, y_test, country_name):
    # Auto-select AR and MA order based on AIC
    try:
        order_res = arma_order_select_ic(y_train, ic='aic', max_ar=2, max_ma=2)
        p, q = order_res.aic_min_order
    except Exception:
        p, q = 1, 0  # fallback
    d = 1  # integration order

    try:
        model = ARIMA(y_train, exog=X_train, order=(p, d, q)).fit()
    except Exception as e:
        print(f"Skipping {country_name}: model fitting failed ({e})")
        return None

    # Train predictions
    train_pred_log = model.predict(start=0, end=len(y_train) - 1, exog=X_train)
    train_pred_real = np.exp(train_pred_log)
    y_train_real = np.exp(y_train)

    # Test predictions
    test_pred_log = model.forecast(steps=len(y_test), exog=X_test)
    test_pred_real = np.exp(test_pred_log)
    y_test_real = np.exp(y_test)

    # Evaluate metrics on train set
    r2_train_log = r2_score(y_train, train_pred_log)
    r2_train_real = r2_score(y_train_real, train_pred_real)
    mae_train_real = mean_absolute_error(y_train_real, train_pred_real)

    # Evaluate metrics on test set
    r2_test_log = r2_score(y_test, test_pred_log)
    r2_test_real = r2_score(y_test_real, test_pred_real)
    mae_test_real = mean_absolute_error(y_test_real, test_pred_real)

    # Create Train set dataframe
    df_train = pd.DataFrame({
        'Country': country_name,
        'Year': pre_covid_fixed[pre_covid_fixed['Country'] == country_name]['Year'].values,
        'Actual_BEV_%': y_train_real.values,
        'Predicted_BEV_%': train_pred_real.values,
        'Type': 'Train',
        'Model_Order': f"({p},{d},{q})"
    })

    # Create Test set dataframe
    df_test = pd.DataFrame({
        'Country': country_name,
        'Year': post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values,
        'Actual_BEV_%': y_test_real.values,
        'Predicted_BEV_%': test_pred_real.values,
        'Type': 'Test',
        'Model_Order': f"({p},{d},{q})"
    })

    return pd.concat([df_train, df_test], ignore_index=True)


In [24]:
# Combining leaders and followers datasets
all_arimax_results_list = []
all_countries = leaders_list + followers_list

for country in all_countries:

    # Data for the specific country to train and test datframe
    # X_tr = train using features from leaders + followers, y_tr = target of leaders + followers
    # X_te = test on using features from leaders + followers, y_te = target of leaders + followers
    X_tr = X_train_pca[pre_covid_fixed['Country'] == country]
    y_tr = y_train[pre_covid_fixed['Country'] == country]
    X_te = X_test_pca[post_covid_fixed['Country'] == country]
    y_te = y_test[post_covid_fixed['Country'] == country]

    # Run both models
    res = run_arimax_model(X_tr, y_tr, X_te, y_te, country)
    all_arimax_results_list.append(res)

# Combine all individual country results
all_arimax_results = pd.concat(all_arimax_results_list, ignore_index=True)

# Calculate Absolute Error
all_arimax_results['Error_Abs'] = (
    all_arimax_results['Actual_BEV_%'] - all_arimax_results['Predicted_BEV_%']
).abs()

# Assign Clusters
all_arimax_results['Cluster'] = all_arimax_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Print results by year and cluster
for year in sorted(all_arimax_results[all_arimax_results['Type'] == 'Test']['Year'].unique()):
    print(f"\n---> ARIMAX MODEL: {int(year)} PREDICTION RESULTS")

    year_data = all_arimax_results[(all_arimax_results['Year'] == year) & (all_arimax_results['Type'] == 'Test')]

    # Leaders Cluster
    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    leaders_data = year_data[year_data['Cluster'] == 'Leaders'].sort_values('Actual_BEV_%', ascending=False)
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))

    # Followers Cluster
    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    followers_data = year_data[year_data['Cluster'] == 'Followers'].sort_values('Actual_BEV_%', ascending=False)
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))

# Save only test results to excel
test_results = all_arimax_results[all_arimax_results['Type'] == 'Test']

# Leaders cluster
leaders_df = test_results[test_results['Cluster'] == 'Leaders'] \
    .sort_values(['Country', 'Year'])

# Followers cluster
followers_df = test_results[test_results['Cluster'] == 'Followers'] \
    .sort_values(['Country', 'Year'])

leaders_df.to_excel("Trial3_ARIMAX_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial3_ARIMAX_Followers_Results.xlsx", index=False)


---> ARIMAX MODEL: 2025 PREDICTION RESULTS

**** LEADERS CLUSTER (2025) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
       Denmark         69.53        92.993959  23.463959
   Netherlands         37.68        50.588552  12.908552
        Sweden         37.58        25.232049  12.347951
       Belgium         35.40        54.792553  19.392553
       Austria         22.34        21.495480   0.844520
        France         21.01        22.462638   1.452638
       Germany         20.08        15.032204   5.047796
         Spain         10.19         8.547276   1.642724
       Hungary          9.36         9.621855   0.261855
        Poland          7.73         1.904231   5.825769
         Italy          7.18         5.570951   1.609049
       Romania          6.59         7.039289   0.449289
Czech Republic          6.55         6.965530   0.415530

**** FOLLOWERS CLUSTER (2025) ****
   Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Finland         37.81        37.